# 04 — SAGE-AVO model and controlled training

| Item | Definition |
|---|---|
| **Scientific purpose** | Train a structure-aware graph/CNN model to refine a supplied low-frequency elastic prior using near/mid/far AVO. |
| **Inputs** | Notebook-03 patch index, train-only normalization, realization splits, AVO, low prior, RGT, elastic targets, segmentation targets, and masks. |
| **Outputs** | Controlled full/no-GNN/no-RGT/no-physics checkpoints, training logs, checkpoint-selection metadata, and run manifests. |
| **Data availability** | Architecture, losses, and orchestration are public; datasets and checkpoints remain private/local. |
| **Local/private-data requirements** | Completed Stage-03 artifacts and adequate PyTorch/PyG compute. No randomly generated data fallback is used. |
| **Software requirements** | `pip install -e ".[ml,notebooks]"` with PyTorch and PyTorch Geometric. |
| **Approximate runtime** | Operator checks: seconds to minutes. Production 120-epoch controlled training: GPU-scale, dependent on hardware. |
| **Pipeline position** | Consumes Notebook 03; produces matched checkpoints and manifests evaluated in Notebook 05. |

The implemented transport is a deterministic straight-path conditional residual flow from the low-frequency prior toward the target. It is **not** a calibrated probabilistic posterior. The graph module is PyTorch Geometric `TransformerConv` graph attention/message passing—not a full-image Vision Transformer.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("Run this notebook from the installed SAGE-AVO repository.")

ROOT = find_repository_root()

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sage_avo.config import load_config, seed_everything
from sage_avo.data import IndexedRealizationPatches
from sage_avo.models import ALL_VARIANTS, LEARNED_VARIANTS, build_sage_avo_variant
from sage_avo.models.sage_avo import angular_features
from sage_avo.training.engine import PhysicsNormalization, train_step
from sage_avo.training.flow import straight_path
from sage_avo.training.losses import LossWeights
from sage_avo.experiments.training import train_controlled_variant

workflow_path = ROOT / "configs" / "sage_avo_s01.yaml"
workflow = load_config(workflow_path)
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError("Create ignored configs/paths.yaml from configs/paths.example.yaml.")
paths = load_config(paths_file)
seed_everything(int(workflow["experiment"]["seed"]))

private_root = Path(paths["private_artifact_root"])
dataset_dir = private_root / "stage_artifacts" / "stage03" / "dataset"
experiment_dir = private_root / "stage_artifacts" / "stage04" / "experiments"
figure_dir = private_root / "figures" / "stage04"
figure_dir.mkdir(parents=True, exist_ok=True)

## 1. Immutable data contract

Each training item contains normalized low/mid/high AVO `[3,H,W]`, normalized low-frequency Vp/Vs/density `[3,H,W]`, normalized elastic target `[3,H,W]`, RGT `[H,W]`, segmentation `[H,W]`, valid mask `[1,H,W]`, and traceability metadata. All controlled variants use the same realization split, patch index, normalization, optimizer schedule, and checkpoint criterion.

In [ ]:
if not (dataset_dir / "dataset_manifest.json").exists():
    raise FileNotFoundError("Run Notebook 03 first; the training notebook never creates a toy dataset.")
dataset_manifest = json.loads((dataset_dir / "dataset_manifest.json").read_text())
normalization = json.loads((dataset_dir / "normalization.json").read_text())
split_ids = json.loads((dataset_dir / "split_ids.json").read_text())
train_data = IndexedRealizationPatches(dataset_dir, "train")
validation_data = IndexedRealizationPatches(dataset_dir, "validation")
sample = train_data[0]
display(pd.Series({
    "train_patches": len(train_data),
    "validation_patches": len(validation_data),
    "split_unit": dataset_manifest["split_unit"],
    "prior_truth_derived": dataset_manifest["prior"]["truth_derived"],
}).to_frame("value"))
print({key: tuple(value.shape) for key, value in sample.items() if isinstance(value, torch.Tensor)})

## 2. Compact AVO summaries

The three stacks are retained as image channels. A least-squares line in `sin²(theta)` additionally yields intercept `P` and gradient `G`; near–2×mid+far supplies curvature. These summaries condition graph edges and node features. They are feature extraction—not the Stage-02 forward model.

In [ ]:
with torch.no_grad():
    angular, gradient = angular_features(sample["avo"].unsqueeze(0))
print("[near, mid, far, P, G, curvature] shape:", tuple(angular.shape))

fig, axes = plt.subplots(1, 5, figsize=(15, 3), constrained_layout=True)
for axis, panel, title in zip(
    axes,
    [sample["avo"][0], sample["avo"][1], sample["avo"][2], angular[0, 3], angular[0, 4]],
    ["Near", "Mid", "Far", "Intercept P", "Gradient G"],
):
    axis.imshow(panel, aspect="auto", cmap="coolwarm")
    axis.set_title(title); axis.set_xticks([]); axis.set_yticks([])
feature_path = figure_dir / "stage04_avo_feature_contract.png"
fig.savefig(feature_path, dpi=300, bbox_inches="tight")
plt.show()

## 3. CNN, RGT-steered graph, and reinjection

1. A CNN encodes the current elastic state, time, three AVO bands, and low-frequency prior.
2. Every image sample is a graph node.
3. Edges include vertical trace neighbors and bidirectional lateral neighbors. For an RGT-steered edge, the adjacent-trace endpoint is chosen within ±3 time samples by minimum RGT mismatch; the no-RGT control uses Cartesian lateral neighbors.
4. Edge weights decrease with local AVO-gradient contrast.
5. Two `TransformerConv` layers perform graph attention/message passing.
6. Graph features are reshaped to the image grid, reinjected into CNN features, and decoded into elastic transport velocity; a second decoder predicts shale/sand/plume classes.

The no-GNN variant retains the CNN and local segmentation decoder. This isolates graph contribution without changing the dataset or target.

In [ ]:
model_config = workflow["model"]
models = {
    variant: build_sage_avo_variant(
        variant,
        hidden_channels=int(model_config["hidden_channels"]),
        graph_layers=int(model_config["graph_layers"]),
        graph_heads=int(model_config["graph_heads"]),
        max_rgt_shift=int(model_config["max_rgt_shift_samples"]),
        classes=int(model_config["classes"]),
    )
    for variant in LEARNED_VARIANTS
}
display(pd.DataFrame([
    {
        "variant": name,
        "graph_mode": model.graph_mode,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
    }
    for name, model in models.items()
]))

full = models["full"].eval()
with torch.no_grad():
    state = sample["low"].unsqueeze(0)
    output = full(state, torch.zeros(1), sample["avo"].unsqueeze(0), state, sample["rgt"].unsqueeze(0))
print("elastic velocity:", tuple(output.velocity.shape))
print("segmentation logits:", tuple(output.segmentation_logits.shape))
print("graph embedding:", tuple(output.embeddings.shape))
print("directed graph edges:", output.edge_indices[0].shape[1])

## 4. Deterministic conditional residual transport

Training samples a time `t ~ Uniform(0,1)` and constructs

\[
x_t=(1-t)x_{low}+t y,\qquad u^*=y-x_{low}.
\]

The network predicts the straight-path velocity conditioned on AVO, the supplied prior, and RGT. Inference starts at `x_low` and integrates the learned velocity from `t=0` to `1` with Heun steps. No stochastic base distribution or posterior calibration is implemented.

In [ ]:
t = torch.tensor([0.35])
state, target_velocity = straight_path(
    sample["low"].unsqueeze(0), sample["target"].unsqueeze(0), t
)
assert torch.allclose(target_velocity, sample["target"].unsqueeze(0) - sample["low"].unsqueeze(0))
print("state and velocity:", tuple(state.shape), tuple(target_velocity.shape))

## 5. Complete training objective

\[
L = w_f L_{flow}+w_p L_{property}+w_s L_{segmentation}
    +w_{phys}L_{Zoeppritz}+w_g L_{graph}.
\]

`L_flow` fits residual velocity; `L_property` supervises the reconstructed full elastic state; segmentation combines weighted cross-entropy and Dice; `L_Zoeppritz` compares observed bands with a differentiable exact-PP forward response from predicted Vp/Vs/density; and `L_graph` penalizes elastic contrast preferentially along high-weight graph edges. The no-physics variant alone sets `w_phys=0`.

In [ ]:
training = workflow["training"]
display(pd.Series(training["loss_weights"], name="weight").to_frame())
display(pd.Series({
    "optimizer": "AdamW",
    "learning_rate": training["learning_rate"],
    "weight_decay": training["weight_decay"],
    "scheduler": "cosine annealing",
    "epochs": training["epochs"],
    "checkpoint_criterion": training["checkpoint_criterion"],
}).to_frame("value"))

## 6. Real-batch operator validation

This cell performs one optimization step on a real Stage-03 batch using the production model and losses. It checks gradients, exact differentiable forward consistency, graph construction, and tensor contracts; it is not reported as a trained result.

In [ ]:
loader = DataLoader(train_data, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
operator_model = build_sage_avo_variant(
    "full",
    hidden_channels=int(model_config["hidden_channels"]),
    graph_layers=int(model_config["graph_layers"]),
    graph_heads=int(model_config["graph_heads"]),
    max_rgt_shift=int(model_config["max_rgt_shift_samples"]),
    classes=int(model_config["classes"]),
)
optimizer = torch.optim.AdamW(operator_model.parameters(), lr=float(training["learning_rate"]))
as_tensor = lambda name: torch.tensor(normalization[name], dtype=torch.float32).view(1, 3, 1, 1)
physics_normalization = PhysicsNormalization(
    x_mean=as_tensor("x_mean"), x_std=as_tensor("x_std"),
    y_mean=as_tensor("y_mean"), y_std=as_tensor("y_std"),
)
weights = LossWeights(**{name: float(value) for name, value in training["loss_weights"].items()})
operator_metrics = train_step(
    operator_model, batch, optimizer, physics_normalization, weights,
    gradient_clip=float(training["gradient_clip"]),
    time_generator=torch.Generator().manual_seed(int(workflow["experiment"]["seed"]) + 17),
)
display(pd.Series(operator_metrics.__dict__).to_frame("one-step value"))
assert all(np.isfinite(value) for value in operator_metrics.__dict__.values())

## 7. Production controlled training

Set `SAGE_AVO_RUN_FULL_TRAINING=1` on suitable compute to train every learned condition. Each run writes a manifest before training, a per-epoch CSV, `best_flow.pt`, `best_sampling.pt`, and `last.pt`. The selected evaluation checkpoint is the shared sampled-validation criterion; seeds and split IDs are embedded in each manifest.

In [ ]:
run_full_training = os.getenv("SAGE_AVO_RUN_FULL_TRAINING", "0") == "1"
run_directories = {}
if run_full_training:
    for variant in LEARNED_VARIANTS:
        run_directories[variant] = train_controlled_variant(
            repository=ROOT,
            config_path=workflow_path,
            config=workflow,
            dataset_directory=dataset_dir,
            experiment_directory=experiment_dir,
            variant=variant,
        )
else:
    print("Production training not requested in this execution.")
    print("Set SAGE_AVO_RUN_FULL_TRAINING=1 to train:", LEARNED_VARIANTS)

status_rows = []
for variant in LEARNED_VARIANTS:
    run_dir = experiment_dir / "runs" / variant
    manifest_file = run_dir / "manifest.json"
    status_rows.append({
        "variant": variant,
        "manifest": manifest_file.exists(),
        "best_sampling_checkpoint": (run_dir / "best_sampling.pt").exists(),
        "status": json.loads(manifest_file.read_text()).get("status") if manifest_file.exists() else "pending",
    })
display(pd.DataFrame(status_rows))

## 8. HCTNet baseline context

HCTNet remains important prior work and is implemented in `sage_avo.models.hctnet`. A headline comparison is intentionally excluded until HCTNet is retrained with the identical truth-derived prior, realization split, normalization, masks, inference tiling, and checkpoint-selection rule. Historical HCTNet results are therefore historical/non-controlled evidence, not entries in the controlled table.

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| `runs/<variant>/best_sampling.pt` | state dictionaries + config/metrics | Checkpoint selected by the common sampled-validation rule | Notebook 05 |
| `training_log.csv` | epoch-level objective terms and validation criteria | Optimization/QC history | Notebook 05 |
| `manifest.json` | seed, split IDs, normalization, prior, commit/config hash | Reproducibility and comparability record | Notebook 05 |

## Scientific checks

- A production-shape real batch passes the CNN, RGT graph, `TransformerConv`, dual decoders, differentiable exact-PP physics loss, and backward optimization.
- The straight-path target is asserted to equal `truth − low prior`.
- Controlled variants differ only in graph mode or physics-loss weight.
- Checkpoint selection, split, normalization, masks, optimizer, and schedule are shared.
- Operator validation is kept distinct from completed model training and scientific performance.

## Next stage

Notebook 05 requires matched `best_sampling.pt` checkpoints for the learned variants. It generates whole-realization predictions, reports per-realization controlled metrics, selects a representative test case by median full-model Vp RMSE, visualizes actual graph edges, and then performs conservative field deployment/QC.